In [7]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("CityMobilityFileStreaming")
    .getOrCreate()
)

DATA_DIR = Path("data/incoming")
OUTPUT_DIR = Path("output/trip_window_counts")
CHECKPOINT_DIR = Path("checkpoints/trip_window_counts")

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.parent.mkdir(parents=True, exist_ok=True)

print("Incoming files:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)
print("Checkpoint folder:", CHECKPOINT_DIR)

Incoming files: data\incoming
Output folder: output\trip_window_counts
Checkpoint folder: checkpoints\trip_window_counts


In [8]:
sample_1 = """trip_id,station_id,event_time
T101,A,2026-10-22 10:01:00
T102,A,2026-10-22 10:02:00
T103,B,2026-10-22 10:04:00
T104,A,2026-10-22 10:06:00
T105,B,2026-10-22 10:08:00
"""

sample_2 = """trip_id,station_id,event_time
T106,A,2026-10-22 10:12:00
T107,B,2026-10-22 10:16:00
T108,A,2026-10-22 10:21:00
T109,B,2026-10-22 10:27:00
T110,A,2026-10-22 10:30:00
"""

(DATA_DIR / "trips_001.csv").write_text(sample_1, encoding="utf-8")
(DATA_DIR / "trips_002.csv").write_text(sample_2, encoding="utf-8")

print("Sample files created.")

Sample files created.


In [9]:
trip_schema = StructType([
    StructField("trip_id", StringType(), True),
    StructField("station_id", StringType(), True),
    StructField("event_time", TimestampType(), True)
])

In [10]:
def read_trip_stream(folder, schema):
    return (
        spark.readStream
        .schema(schema)
        .option("header", True)
        .option("maxFilesPerTrigger", 1)
        .csv(str(folder))
    )

trips_stream = read_trip_stream(DATA_DIR, trip_schema)

In [11]:
def build_window_counts(stream_df, window_size="5 minutes", watermark_delay="10 minutes"):
    return (
        stream_df
        .withWatermark("event_time", watermark_delay)
        .groupBy(
            F.window("event_time", window_size),
            F.col("station_id")
        )
        .count()
        .select(
            F.col("window.start").alias("window_start"),
            F.col("window.end").alias("window_end"),
            "station_id",
            F.col("count").alias("trip_count")
        )
    )

window_counts = build_window_counts(trips_stream)

In [12]:
def start_available_now_query(result_df, output_folder, checkpoint_folder):
    return (
        result_df.writeStream
        .format("parquet")
        .outputMode("append")
        .option("path", str(output_folder))
        .option("checkpointLocation", str(checkpoint_folder))
        .trigger(availableNow=True)
        .start()
    )

query = start_available_now_query(
    window_counts,
    OUTPUT_DIR,
    CHECKPOINT_DIR
)

query.awaitTermination()

print("Streaming run finished.")
print("Parquet output:", OUTPUT_DIR)

Streaming run finished.
Parquet output: output\trip_window_counts
